# 04a - TabFM Modeling

This notebook evaluates Google's TabFM tabular foundation model on the same processed full-feature dataset and stratified holdout split used by the XGBoost notebook. TabFM performs zero-shot tabular classification: the training rows are provided as context, and the pretrained PyTorch model predicts holdout churn probabilities without dataset-specific gradient training or hyperparameter tuning.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

from churn_ml.models.evaluate_model import classification_metrics

In [2]:
# fast completing parameters
# RANDOM_STATE = 42
# TARGET_COLUMN = "Churn Value"
# CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
# TABFM_N_ESTIMATORS = 8
# TABFM_BATCH_SIZE = 1
# TABFM_MAX_CONTEXT_ROWS = 1024


RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
TABFM_N_ESTIMATORS = 8
TABFM_BATCH_SIZE = 1
TABFM_MAX_CONTEXT_ROWS = 2048
TABFM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

def positive_class_probability(classifier, X):
    class_labels = list(classifier.classes_)
    if 1 not in class_labels:
        raise ValueError(f"Expected positive class label 1 in {class_labels}.")
    return classifier.predict_proba(X)[:, class_labels.index(1)]

DATA_PATH = find_project_file(Path("data/processed/prediction_df_xgboost.csv"))
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))
PROJECT_ROOT = DATA_PATH.parents[2]
load_dotenv(PROJECT_ROOT / ".env", override=False)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
HF_TOKEN_CONFIGURED = bool(os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN"))
MLFLOW_EXPERIMENT_NAME = "telco-churn-modeling"
mlflow.set_tracking_uri("sqlite:///" + (PROJECT_ROOT / "mlflow.db").as_posix())
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
TABFM_CONFIG_PATH = PROJECT_ROOT / "artifacts/models/tabfm_pytorch_config.json"
TABFM_REPO_ID = "google/tabfm-1.0.0-pytorch"
TABFM_MODEL_TYPE = "classification"
TABFM_CHECKPOINT_DIR = PROJECT_ROOT / "artifacts/models/tabfm_checkpoint" / TABFM_MODEL_TYPE
TABFM_CHECKPOINT_PATH = TABFM_CHECKPOINT_DIR / "pytorch_model.bin"

def prepare_tabfm_checkpoint(model_type=TABFM_MODEL_TYPE):
    checkpoint_dir = PROJECT_ROOT / "artifacts/models/tabfm_checkpoint" / model_type
    checkpoint_path = checkpoint_dir / "pytorch_model.bin"
    if checkpoint_path.exists():
        print(f"Using existing converted TabFM checkpoint: {checkpoint_path}")
        return checkpoint_path

    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN") or None
    snapshot_path = Path(snapshot_download(
        repo_id=TABFM_REPO_ID,
        allow_patterns=[f"{model_type}/model.safetensors", f"{model_type}/config.json", "config.json"],
        token=token,
    ))
    safetensors_path = snapshot_path / model_type / "model.safetensors"
    if not safetensors_path.exists():
        raise FileNotFoundError(f"TabFM safetensors checkpoint not found at: {safetensors_path}")

    state_dict = load_file(str(safetensors_path), device="cpu")
    torch.save(state_dict, checkpoint_path)
    print(f"Converted TabFM safetensors checkpoint to: {checkpoint_path}")
    return checkpoint_path

In [3]:

model_df = pd.read_csv(DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert model_df.columns[-1] == TARGET_COLUMN, "The target must be the final column."
X = model_df.drop(columns=TARGET_COLUMN)
y = model_df[TARGET_COLUMN]
if len(value_df) != len(model_df) or not value_df[TARGET_COLUMN].equals(y):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")

customer_ltv = value_df.set_index("CustomerID")["CLTV"].rename("predicted_ltv_if_retained")
customer_ids = value_df["CustomerID"]
X_train, X_test, y_train, y_test, ltv_train, ltv_test, customer_id_train, customer_id_test = train_test_split(
    X, y, customer_ltv, customer_ids, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")

print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Prediction threshold: {decision_threshold:.1%}")
print(f"Torch version: {torch.__version__}")
print(f"TabFM device: {TABFM_DEVICE}")
print(f"Hugging Face token configured: {HF_TOKEN_CONFIGURED}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Training rows: 5,634; test rows: 1,409
Prediction threshold: 26.5%
Torch version: 2.11.0+cu128
TabFM device: cuda
Hugging Face token configured: True
CUDA version: 12.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Zero-shot TabFM inference

TabFM is loaded with the PyTorch backend and, when available, runs on CUDA. The `.fit()` step prepares TabFM's tabular context and encoders; it does not tune model weights on this project dataset. The first run may download the TabFM weights from Hugging Face. If `HF_TOKEN` is set in the project-root `.env`, the notebook loads it before requesting the checkpoint. The current Hugging Face checkpoint is published as `model.safetensors`; this notebook converts it once to the `pytorch_model.bin` file expected by TabFM 1.0.0 and reuses the converted file on later runs.

The default notebook setting uses `TABFM_N_ESTIMATORS = 4` and `TABFM_MAX_CONTEXT_ROWS = 1024` so the first run is bounded on an 8 GB laptop GPU. Increase these after the workflow completes successfully if you want a slower, stronger TabFM pass.

In [4]:
if TABFM_DEVICE == "cuda":
    torch.cuda.empty_cache()

tabfm_checkpoint_path = prepare_tabfm_checkpoint()
tabfm_backbone = tabfm_v1_0_0.load(checkpoint_path=str(tabfm_checkpoint_path), model_type=TABFM_MODEL_TYPE, device=TABFM_DEVICE)
tabfm_model = TabFMClassifier(
    model=tabfm_backbone,
    n_estimators=TABFM_N_ESTIMATORS,
    batch_size=TABFM_BATCH_SIZE,
    max_num_rows=TABFM_MAX_CONTEXT_ROWS,
    random_state=RANDOM_STATE,
    use_amp=(TABFM_DEVICE == "cuda"),
    verbose=True,
)
tabfm_model.fit(X_train, y_train)
print("TabFM fit complete.")

Using existing converted TabFM checkpoint: W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\tabfm_checkpoint\classification\pytorch_model.bin
Columns classified as categorical: []
Columns classified as continuous: ['Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Paperless Billing', 'Monthly Charges', 'Total Charges', 'Total_Charges_Missing', 'Multiple Lines_No', 'Multiple Lines_No phone service', 'Multiple Lines_Yes', 'Internet Service_DSL', 'Internet Service_Fiber optic', 'Internet Service_No', 'Online Security_No', 'Online Security_No internet service', 'Online Security_Yes', 'Online Backup_No', 'Online Backup_No internet service', 'Online Backup_Yes', 'Device Protection_No', 'Device Protection_No internet service', 'Device Protection_Yes', 'Tech Support_No', 'Tech Support_No internet service', 'Tech Support_Yes', 'Streaming TV_No', 'Streaming TV_No internet service', 'Streaming TV_Yes', 'Streaming Movi

In [5]:
y_proba = positive_class_probability(tabfm_model, X_test)
y_pred = (y_proba >= decision_threshold).astype(int)

tabfm_metrics = pd.Series(classification_metrics(y_test, y_pred, y_proba), name="tabfm")
display(tabfm_metrics.to_frame())

TABFM_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
tabfm_config = {
    "model": "tabfm",
    "backend": "pytorch",
    "checkpoint": TABFM_REPO_ID,
    "checkpoint_path": str(tabfm_checkpoint_path),
    "device": TABFM_DEVICE,
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "n_estimators": TABFM_N_ESTIMATORS,
    "batch_size": TABFM_BATCH_SIZE,
    "max_context_rows": TABFM_MAX_CONTEXT_ROWS,
    "random_state": RANDOM_STATE,
    "holdout_metrics": tabfm_metrics.to_dict(),
}
with TABFM_CONFIG_PATH.open("w", encoding="utf-8") as file:
    json.dump(tabfm_config, file, indent=2)
print(f"Saved TabFM configuration and holdout metrics to {TABFM_CONFIG_PATH}")

,tabfm
accuracy,0.762952
precision,0.536101
recall,0.794118
f1,0.640086
pr_auc,0.678069
roc_auc,0.859865


Saved TabFM configuration and holdout metrics to W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\tabfm_pytorch_config.json


## Holdout confusion matrix

The classification threshold defaults to the training-set churn rate for consistency with the other modeling notebooks.

In [6]:
confusion = confusion_matrix(y_test, y_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion,
    x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues",
    text=confusion,
    texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"TabFM Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

# Expected Value of Retention Targeting

This evaluation retrieves the raw dataset's `CLTV` value for every holdout-test customer and treats it as that customer's **predicted lifetime value if retained**. `CLTV` is deliberately not a churn-model feature; it is used only after prediction to prioritize outreach.

The expected net value for each customer is calculated as:

$$P(\text{churn}) \times 10\% \times \text{predicted LTV if retained} - \$20 - (40\% \times \$500)$$

Assumptions: outreach costs **$20** for every targeted customer. The offer costs **$500** only when it is accepted; this scenario assumes a **40% offer-acceptance rate among targeted customers**, including customers who would have stayed without outreach. Its expected cost is therefore $200 per target. The 10% retention uplift is a separate scenario assumption: it represents the share of would-be churners saved by the intervention. Neither assumption is estimated by the churn model, so replace them when campaign data becomes available. The top 100 are selected by expected net value, not merely by churn probability, so high-value customers are prioritized.

In [7]:
OUTREACH_COST = 20
OFFER_COST = 500
RETENTION_UPLIFT = 0.10
OFFER_ACCEPTANCE_RATE = 0.40
TARGET_COUNT = 100

targeting_candidates = pd.DataFrame({
    "CustomerID": customer_id_test.to_numpy(),
    "predicted_churn_probability": y_proba,
    "predicted_ltv_if_retained": ltv_test.to_numpy(),
})
targeting_candidates["expected_value_before_cost"] = (
    targeting_candidates["predicted_churn_probability"]
    * RETENTION_UPLIFT
    * targeting_candidates["predicted_ltv_if_retained"]
)
targeting_candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
targeting_candidates["campaign_cost"] = OUTREACH_COST + targeting_candidates["expected_offer_cost"]
targeting_candidates["expected_net_value"] = (
    targeting_candidates["expected_value_before_cost"]
    - targeting_candidates["campaign_cost"]
)

top_100_targets = (
    targeting_candidates
    .sort_values("expected_net_value", ascending=False)
    .head(TARGET_COUNT)
    .reset_index(drop=True)
)

targeting_summary = pd.DataFrame({
    "customers_targeted": [len(top_100_targets)],
    "expected_value_before_cost": [top_100_targets["expected_value_before_cost"].sum()],
    "outreach_cost": [OUTREACH_COST * len(top_100_targets)],
    "expected_offer_cost": [top_100_targets["expected_offer_cost"].sum()],
    "campaign_cost": [top_100_targets["campaign_cost"].sum()],
    "expected_net_value": [top_100_targets["expected_net_value"].sum()],
})
display(targeting_summary.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))
top_100_targets.style.format({
    "predicted_churn_probability": "{:.1%}",
    "predicted_ltv_if_retained": "${:,.0f}",
    "expected_value_before_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
})

,customers_targeted,expected_value_before_cost,outreach_cost,expected_offer_cost,campaign_cost,expected_net_value
0,100,"$38,017.86","$2,000.00","$20,000.00","$22,000.00","$16,017.86"


,CustomerID,predicted_churn_probability,predicted_ltv_if_retained,expected_value_before_cost,expected_offer_cost,campaign_cost,expected_net_value
0,0295-PPHDO,94.7%,"$5,962",$564.76,$200.00,$220.00,$344.76
1,5178-LMXOP,96.0%,"$5,795",$556.09,$200.00,$220.00,$336.09
2,7180-PISOG,92.5%,"$5,554",$513.72,$200.00,$220.00,$293.72
3,1628-BIZYP,87.6%,"$5,754",$504.18,$200.00,$220.00,$284.18
4,3716-BDVDB,86.3%,"$5,795",$500.01,$200.00,$220.00,$280.01
5,1320-HTRDR,80.5%,"$5,948",$478.55,$200.00,$220.00,$258.55
6,6023-YEBUP,89.4%,"$5,351",$478.23,$200.00,$220.00,$258.23
7,2865-TCHJW,82.3%,"$5,808",$478.11,$200.00,$220.00,$258.11
8,7409-KIUTL,86.4%,"$5,345",$461.78,$200.00,$220.00,$241.78
9,3006-XIMLN,86.9%,"$5,299",$460.25,$200.00,$220.00,$240.25


## MLflow tracking

This section records the lightweight experiment evidence for notebook 04: TabFM backend/device metadata, holdout metrics, and retention-targeting outputs. It intentionally does not log the TabFM model checkpoint or weights.

In [8]:
with mlflow.start_run(run_name="04_tabfm_modeling"):
    mlflow.set_tags({
        "notebook": "04a_tabFM_modeling.ipynb",
        "model_family": "tabfm",
        "stage": "candidate_modeling",
    })
    mlflow.log_params({
        "random_state": RANDOM_STATE,
        "target_column": TARGET_COLUMN,
        "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "threshold_policy": "training_churn_rate" if CHURN_THRESHOLD is None else "manual",
        "decision_threshold": decision_threshold,
        "backend": "pytorch",
        "checkpoint": TABFM_REPO_ID,
        "checkpoint_path": str(TABFM_CHECKPOINT_PATH),
        "device": TABFM_DEVICE,
        "torch_version": torch.__version__,
        "torch_cuda_version": torch.version.cuda,
        "n_estimators": TABFM_N_ESTIMATORS,
        "batch_size": TABFM_BATCH_SIZE,
        "max_context_rows": TABFM_MAX_CONTEXT_ROWS,
        "use_amp": TABFM_DEVICE == "cuda",
    })
    if torch.cuda.is_available():
        mlflow.log_param("gpu_name", torch.cuda.get_device_name(0))
    mlflow.log_metrics({f"holdout_{key}": float(value) for key, value in tabfm_metrics.to_dict().items()})
    mlflow.log_metrics({
        "targeting_expected_net_value": float(targeting_summary.loc[0, "expected_net_value"]),
        "targeting_expected_value_before_cost": float(targeting_summary.loc[0, "expected_value_before_cost"]),
        "targeting_campaign_cost": float(targeting_summary.loc[0, "campaign_cost"]),
        "targeting_customers_targeted": int(targeting_summary.loc[0, "customers_targeted"]),
    })
    mlflow.log_dict(tabfm_config, "configs/tabfm_pytorch_config.json")
    mlflow.log_table(tabfm_metrics.reset_index().rename(columns={"index": "metric", "tabfm": "value"}), "tables/holdout_metrics.json")
    mlflow.log_table(targeting_summary, "tables/targeting_summary.json")
    mlflow.log_table(top_100_targets, "tables/top_100_targets.json")

print(f"Logged MLflow run to {PROJECT_ROOT / 'mlflow.db'}")

Logged MLflow run to W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\mlflow.db


## TabFM notes

TabFM model weights are released under Google's TabFM non-commercial license. Evaluate this notebook's holdout results, memory use, and license fit before using TabFM outside this learning project.